<a href="https://colab.research.google.com/github/gechsimxx-design/BCO7006---ASM-4-/blob/kevin/ASM_4_Group_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Cell 1: Import libraries and display settings**

In [8]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 105)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
sns.set()

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

**What it does:**\
This cell imports the main Python libraries used for data analysis, visualisation and statistical checking.

**Libraries used:**\
numpy is used for numerical calculations, especially log transformation.
pandas is used to load, clean and manage the data as tables.
matplotlib.pyplot and seaborn are used to create charts.
scipy.stats is used later to calculate Pearson correlation between house features and sale price.
warnings is used to hide warning messages so the notebook output is easier to read.

**pandas / numpy methods:**\
pd.set_option('max_columns', 105) changes the pandas display setting so that up to 105 columns can be shown when viewing the dataset.

**Loops / conditions:**\
There are no loops or conditions in this cell.

**Why it matters for the business:**\
This cell sets up the tools needed to inspect the housing data, clean it and identify patterns that may help predict house prices.

###**Cell 2: Set modelling options**

In [9]:
nr_cv = 5

use_logvals = 1
target = 'SalePrice_Log'

min_val_corr = 0.4

drop_similar = 1

**What it does:**\
This cell sets important options used later in the notebook.

**Libraries used:**\
No external library is directly used in this cell. It creates Python variables.

**pandas / numpy methods:**\
No pandas or numpy methods are used in this cell.

**Loops / conditions:**\
There are no loops in this cell. However, drop_similar = 1 is used later in an if condition to decide whether similar highly correlated features should be removed.

**Explanation:**\
nr_cv = 5 means the notebook will use five-fold cross-validation when testing models.
use_logvals = 1 means the notebook will use log-transformed values.
target = 'SalePrice_Log' sets the prediction target as the log-transformed sale price.
min_val_corr = 0.4 means only features with correlation above 0.4 are treated as strong predictors.
drop_similar = 1 means the notebook will remove some features that repeat similar information.

**Why it matters for the business:**\
These settings control how strict the model is when selecting useful property features for price prediction.

###**Cell 3: Define helper function for model score**

In [10]:
def get_best_score(grid):

    best_score = np.sqrt(-grid.best_score_)
    print(best_score)
    print(grid.best_params_)
    print(grid.best_estimator_)

    return best_score

**What it does:**\
This function extracts and prints the best model score, best parameters and best model from a GridSearchCV result.

**Libraries used:**\
numpy is used through np.sqrt().

**pandas / numpy methods:**\
np.sqrt() calculates the square root. The notebook uses negative mean squared error during model testing, so -grid.best_score_ converts it back into a positive value before calculating RMSE.

**Loops / conditions:**\
There are no loops or conditions in this function.

**Why it matters for the business:**\
This function helps compare models using RMSE. A lower RMSE means the predicted house prices are closer to the actual prices.

###**Cell 4: Define helper function for correlation matrix**

In [11]:
def plot_corr_matrix(df, nr_c, targ) :

    corr = df.corr()
    corr_abs = corr.abs()
    cols = corr_abs.nlargest(nr_c, targ)[targ].index
    cm = np.corrcoef(df[cols].values.T)

    plt.figure(figsize=(nr_c/1.5, nr_c/1.5))
    sns.set(font_scale=1.25)
    sns.heatmap(cm, linewidths=1.5, annot=True, square=True,
                fmt='.2f', annot_kws={'size': 10},
                yticklabels=cols.values, xticklabels=cols.values
               )
    plt.show()

**What it does:**\
This function creates a heatmap showing the strongest correlations between selected features and the target variable.

**Libraries used:**\
pandas is used to calculate correlations.
numpy is used to create the correlation matrix.
matplotlib and seaborn are used to create the heatmap.

**pandas / numpy methods:**
* df.corr() calculates correlation between numerical columns.
* .abs() converts correlation values into positive numbers so strong negative and positive relationships can both be compared.
* .nlargest() selects the strongest correlations.
* .index returns the selected column names.
* np.corrcoef() calculates the correlation coefficient matrix.
* .values.T converts the selected DataFrame values into a transposed numerical array.

Loops / conditions:
There are no explicit loops or conditions in this function.

Why it matters for the business:
This helps identify which house features are most strongly connected with sale price, such as quality, living area, garage capacity and basement size.

###**Cell 5: Load the train and test datasets**

In [12]:
df_train = pd.read_csv("https://raw.githubusercontent.com/gechsimxx-design/BCO7006---ASM-4-/refs/heads/main/train.csv")
df_test = pd.read_csv("https://raw.githubusercontent.com/gechsimxx-design/BCO7006---ASM-4-/refs/heads/main/test.csv")

**What it does:**\
This cell loads the training and testing datasets into Python.

**Libraries used:**\
pandas is used to read the CSV files and store them as DataFrames.

**pandas / numpy methods:**\
pd.read_csv() reads a CSV file and converts it into a pandas DataFrame. A DataFrame is a table with rows and columns.

**Loops / conditions:**\
There are no loops or conditions in this cell.

**Why it matters for the business:**\
This is the starting point of the analysis. The training data is used to learn from past house sales, while the testing data is used to predict unknown sale prices.

###**Cell 6: Check dataset size and structure**

In [13]:
print(df_train.shape)
print("*"*50)
print(df_test.shape)

print(df_train.info())
print("*"*50)
print(df_test.info())

(1460, 81)
**************************************************
(1459, 80)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non

**What it does:**\
This cell checks the number of rows and columns in the training and testing datasets. It also shows column names, data types and missing values.

**Libraries used:**\
pandas is used because df_train and df_test are DataFrames.

**pandas / numpy methods:**\
* df_train.shape returns the number of rows and columns in the training data.
* df_test.shape returns the number of rows and columns in the testing data.
* .info() shows the column names, data types and number of non-missing values.

**Loops / conditions:**\
There are no loops or conditions in this cell.

**Why it matters for the business:**\
This confirms that the data has loaded correctly. It also shows that the training data has SalePrice, while the test data does not. This is important because the model must learn from known sale prices and then predict unknown ones.

###**Cell 7: Preview and summarise the data**

In [14]:
df_train.head()
df_train.describe()


df_test.head()
df_test.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
count,1459.000000,1459.000000,1232.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1444.000000,1458.000000,1458.000000,1458.000000,1458.000000,1459.000000,1459.000000,1459.000000,1459.000000,1457.000000,1457.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.00000,1381.000000,1458.000000,1458.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000,1459.000000
mean,2190.000000,57.378341,68.580357,9819.161069,6.078821,5.553804,1971.357779,1983.662783,100.709141,439.203704,52.619342,554.294925,1046.117970,1156.534613,325.967786,3.543523,1486.045922,0.434454,0.065202,1.570939,0.377656,2.854010,1.042495,6.385195,0.58122,1977.721217,1.766118,472.768861,93.174777,48.313914,24.243317,1.794380,17.064428,1.744345,58.167923,6.104181,2007.769705
std,421.321334,42.746880,22.376841,4955.517327,1.436812,1.113740,30.390071,21.130467,177.625900,455.268042,176.753926,437.260486,442.898624,398.165820,420.610226,44.043251,485.566099,0.530648,0.252468,0.555190,0.503017,0.829788,0.208472,1.508895,0.64742,26.431175,0.775945,217.048611,127.744882,68.883364,67.227765,20.207842,56.609763,30.491646,630.806978,2.722432,1.301740
min,1461.000000,20.000000,21.000000,1470.000000,1.000000,1.000000,1879.000000,1950.000000,0.000000,0.000000,0.000000,0.000000,0.000000,407.000000,0.000000,0.000000,407.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.00000,1895.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000
25%,1825.500000,20.000000,58.000000,7391.000000,5.000000,5.000000,1953.000000,1963.000000,0.000000,0.000000,0.000000,219.250000,784.000000,873.500000,0.000000,0.000000,1117.500000,0.000000,0.000000,1.000000,0.000000,2.000000,1.000000,5.000000,0.00000,1959.000000,1.000000,318.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000
50%,2190.000000,50.000000,67.000000,9399.000000,6.000000,5.000000,1973.000000,1992.000000,0.000000,350.500000,0.000000,460.000000,988.000000,1079.000000,0.000000,0.000000,1432.000000,0.000000,0.000000,2.000000,0.000000,3.000000,1.000000,6.000000,0.00000,1979.000000,2.000000,480.000000,0.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000
75%,2554.500000,70.000000,80.000000,11517.500000,7.000000,6.000000,2001.000000,2004.000000,164.000000,753.500000,0.000000,797.750000,1305.000000,1382.500000,676.000000,0.000000,1721.000000,1.000000,0.000000,2.000000,1.000000,3.000000,1.000000,7.000000,1.00000,2002.000000,2.000000,576.000000,168.000000,72.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000
max,2919.000000,190.000000,200.000000,56600.000000,10.000000,9.000000,2010.000000,2010.000000,1290.000000,4010.000000,1526.000000,2140.000000,5095.000000,5095.000000,1862.000000,1064.000000,5095.000000,3.000000,2.000000,4.000000,2.000000,6.000000,2.000000,15.000000,4.00000,2207.000000,5.000000,1488.000000,1424.000000,742.000000,1012.000000,360.000000,576.000000,800.000000,17000.000000,12.000000,2010.000000


**What it does:**\
This cell displays sample rows and summary statistics for the training and testing datasets.

**Libraries used:**\
pandas is used to view and summarise the DataFrames.

**pandas / numpy methods:**
* .head() displays the first five rows of the dataset.
* .describe() returns summary statistics for numerical columns, including count, mean, standard deviation, minimum, quartiles and maximum.

**Loops / conditions:**\
There are no loops or conditions in this cell.

**Why it matters for the business:**\
This helps the analyst understand the range of house features and prices. For example, it shows the average sale price, the lowest sale price and the highest sale price in the training data.